# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports

import os
import html
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display, update_display
import ipywidgets as widgets
from openai import OpenAI

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

openai = OpenAI()

In [ ]:
# Here is the System Prompt

system_prompt = """
You are an assistant who will provide the answer to the technical question asked to you.
"""

In [ ]:
# Here is the function for QnA

def stream_QnA_Gpt():
    text_box = widgets.Text(description="Your Question:")
    button = widgets.Button(description="Submit")

    # This is displayed once and updated in place.
    answer_box = widgets.HTML(
        value="",
        layout=widgets.Layout(
            white_space="pre-wrap",
            border="1px solid #ddd",
            padding="10px",
            width="100%",
        ),
    )

    display(text_box, button, answer_box)

    def on_button_clicked(b):
        button.disabled = True
        answer_box.value = "Generating…"

        try:
            response = ""

            stream = openai.chat.completions.create(
                model=MODEL_GPT,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": text_box.value},
                ],
                stream=True,
            )

            for chunk in stream:
                if not chunk.choices:
                    continue

                content = chunk.choices[0].delta.content
                if content:
                    response += content
                    #response = Markdown(response)
                    answer_box.value = html.escape(response).replace("\n", "<br>").replace("**","")

        except Exception as e:
            answer_box.value = f"<b>Error:</b> {html.escape(str(e))}"

        finally:
            button.disabled = False

    button.on_click(on_button_clicked)

In [ ]:
# Get Llama 3.2 to answer
stream_QnA_Gpt()